# Experiment: overfitting check for Baseline and Stacked
Minimal copy of the Task 1-5 code of `Deliverable.ipynb` with the Task 5 setup (`A_FACTOR = 4`, `pos_weight`, 3 seeds, 20 epochs). The training is repeated to log, after every epoch, the macro F1 on the training set (original clauses, without the oversampled copies, threshold 0.5) and the validation loss (same weighted loss as training). The curves go to `Experiments/results/task5_overfitting_curves.pkl`. Not part of the submission.

In [1]:
ROOT = Path.cwd().parents[1]  # this notebook lives in Experiments/notebooks
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

data_folder = ROOT / 'Data' / 'ToS-100'
keep = ['document_ID', 'text', 'A', 'CH', 'CR', 'LTD', 'TER']
training_set = pd.read_csv(data_folder / 'train.csv')[keep]
test_set = pd.read_csv(data_folder / 'test.csv')[keep]
val_set = pd.read_csv(data_folder / 'validation.csv')[keep]

label_cols = ['A', 'CH', 'CR', 'LTD', 'TER']
y_val = val_set[label_cols].to_numpy(dtype=np.float32)
y_test = test_set[label_cols].to_numpy(dtype=np.float32)

In [2]:
import nltk
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

ENUMERATOR = r"(?:\d+|[a-z]|[ivx]+)"
NUMBERING_RE = re.compile(
    rf"^(?:-l[rs]b- {ENUMERATOR} -r[rs]b-"
    rf"|{ENUMERATOR} -r[rs]b-"
    rf"|\d+(?:\.\d+)+\.?"
    rf"|\d+ [.°]"
    rf"|[•·●*-])\s+"
)
TOKEN_MAP = {
    '-lrb-': '(', '-rrb-': ')', '-lsb-': '[', '-rsb-': ']', '-lcb-': '{', '-rcb-': '}',
    '``': '"', "''": '"', '`': "'",
}
CLITICS = {"'s", "'re", "'ve", "'ll", "'d", "'m", "n't"}
SPECIAL_CHARS_RE = re.compile(r"[^a-z0-9\s.,;:!?()\[\]{}\"'/$%&@-]")

lemmatizer = WordNetLemmatizer()


def get_wordnet_key(tag: str) -> str | None:
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    return None


def strip_numbering(text: str) -> str:
    while (stripped := NUMBERING_RE.sub('', text, count=1)) != text:
        text = stripped
    return text


def lemmatize(tokens: list[str]) -> list[str]:
    lemmas = []
    for i, (word, tag) in enumerate(pos_tag(tokens)):
        key = get_wordnet_key(tag)
        before_negation = i + 1 < len(tokens) and tokens[i + 1] == "n't"
        lemmas.append(lemmatizer.lemmatize(word, key) if key and not before_negation else word)
    return lemmas


def reattach_clitics(tokens: list[str]) -> list[str]:
    merged = []
    for token in tokens:
        if token in CLITICS and merged:
            merged[-1] += token
        else:
            merged.append(token)
    return merged


def clean_text(text: str) -> str:
    text = strip_numbering(text)
    tokens = [TOKEN_MAP.get(token, token) for token in text.split()]
    tokens = reattach_clitics(lemmatize(tokens))
    text = SPECIAL_CHARS_RE.sub(' ', ' '.join(tokens))
    return ' '.join(text.split())


for df in (training_set, val_set, test_set):
    df['clean_text'] = df['text'].apply(clean_text)

In [3]:
import gensim.downloader as gloader

glove = gloader.load('glove-wiki-gigaword-100')

PAD, UNK = '<PAD>', '<UNK>'
PAD_IDX, UNK_IDX = 0, 1

train_counts = Counter(token for text in training_set['clean_text'] for token in text.split())
train_oov = [token for token in train_counts if token not in glove.key_to_index]

idx_to_word = [PAD, UNK] + glove.index_to_key + train_oov
word_to_idx = {word: idx for idx, word in enumerate(idx_to_word)}

def compose_vector(token: str) -> np.ndarray | None:
    clitic = next((c for c in CLITICS if token.endswith(c) and token != c), None)
    stem = token[:-len(clitic)] if clitic else token
    parts = re.split(r'[-/]', stem) + ([clitic] if clitic else [])
    vectors = [glove[part] for part in parts if part in glove.key_to_index]
    return np.mean(vectors, axis=0) if vectors else None


rng = np.random.default_rng(42)
embedding_dim = glove.vector_size

embedding_matrix = np.zeros((len(idx_to_word), embedding_dim), dtype=np.float32)
embedding_matrix[UNK_IDX] = rng.uniform(-0.05, 0.05, embedding_dim)
embedding_matrix[2:2 + len(glove.index_to_key)] = glove.vectors

composed = []
for token in train_oov:
    vector = compose_vector(token)
    if vector is None:
        vector = rng.uniform(-0.05, 0.05, embedding_dim)
    else:
        composed.append(token)
    embedding_matrix[word_to_idx[token]] = vector

def encode(text: str) -> list[int]:
    return [word_to_idx.get(token, UNK_IDX) for token in text.split()]


for df in (training_set, val_set, test_set):
    df['token_ids'] = df['clean_text'].apply(encode)

In [4]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset

MAX_LEN = 150


class ClauseDataset(Dataset):
    def __init__(self, token_ids: pd.Series, labels: np.ndarray):
        self.sequences = [torch.tensor((ids or [UNK_IDX])[:MAX_LEN]) for ids in token_ids]
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


def collate(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences])
    padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    return padded, lengths, torch.stack(labels)


import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class BiLSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix: np.ndarray, hidden_size: int, num_layers: int, n_labels: int = len(label_cols),
                 freeze_embedding: bool = False):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=freeze_embedding, padding_idx=PAD_IDX
        )
        self.lstm = nn.LSTM(embedding_matrix.shape[1], hidden_size, num_layers=num_layers,
                            batch_first=True, bidirectional=True)
        self.dense = nn.Linear(2 * hidden_size, n_labels)

    def forward(self, token_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        packed = pack_padded_sequence(self.embedding(token_ids), lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        outputs, _ = self.lstm(packed)
        states, _ = pad_packed_sequence(outputs, batch_first=True)
        padding = torch.arange(states.size(1), device=states.device) >= lengths.to(states.device).unsqueeze(1)
        return self.dense(states.masked_fill(padding.unsqueeze(-1), float('-inf')).max(dim=1).values)

In [5]:
HIDDEN_SIZE = 128
SEEDS = [42, 1337, 2026]
EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


from torch.utils.data import DataLoader

A_FACTOR = 4

a_clauses = training_set[training_set['A'] == 1]
train_resampled = pd.concat([training_set] + [a_clauses] * (A_FACTOR - 1), ignore_index=True)
y_resampled = train_resampled[label_cols].to_numpy(dtype=np.float32)

n_pos = y_resampled.sum(axis=0)
pos_weight = torch.tensor((len(y_resampled) - n_pos) / n_pos)

train_dataset = ClauseDataset(train_resampled['token_ids'], y_resampled)
val_loader = DataLoader(ClauseDataset(val_set['token_ids'], y_val), batch_size=256, collate_fn=collate)
test_loader = DataLoader(ClauseDataset(test_set['token_ids'], y_test), batch_size=256, collate_fn=collate)
y_train_orig = training_set[label_cols].to_numpy(dtype=np.float32)
train_eval_loader = DataLoader(ClauseDataset(training_set['token_ids'], y_train_orig), batch_size=256, collate_fn=collate)


from sklearn.metrics import f1_score, precision_score, recall_score

THRESHOLD_GRID = np.arange(0.05, 0.96, 0.05)


def compute_metrics(y_true: np.ndarray, probs: np.ndarray, thresholds: float | np.ndarray = 0.5) -> dict:
    y_pred = (probs >= thresholds).astype(int)
    per_category = f1_score(y_true, y_pred, average=None, zero_division=0)
    return {
        'macro F1': per_category.mean(),
        'macro precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        **{f'F1 {label}': score for label, score in zip(label_cols, per_category)},
    }


def tune_thresholds(y_true: np.ndarray, probs: np.ndarray) -> np.ndarray:
    thresholds = []
    for c in range(y_true.shape[1]):
        scores = [f1_score(y_true[:, c], probs[:, c] >= t, zero_division=0) for t in THRESHOLD_GRID]
        thresholds.append(THRESHOLD_GRID[np.argmax(scores)])
    return np.array(thresholds)


@torch.no_grad()
def predict_proba(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    return np.concatenate([torch.sigmoid(model(ids.to(DEVICE), lengths)).cpu().numpy() for ids, lengths, _ in loader])


@torch.no_grad()
def predict_logits(model: nn.Module, loader: DataLoader) -> torch.Tensor:
    model.eval()
    return torch.cat([model(ids.to(DEVICE), lengths).cpu() for ids, lengths, _ in loader])


def train_model(num_layers: int, seed: int) -> dict:
    set_seed(seed)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                              generator=torch.Generator().manual_seed(seed))

    model = BiLSTMClassifier(embedding_matrix, HIDDEN_SIZE, num_layers).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_f1, best_epoch, best_state = -1.0, 0, None
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for ids, lengths, labels in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(ids.to(DEVICE), lengths), labels.to(DEVICE))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(labels)

        val_logits = predict_logits(model, val_loader)
        val_f1 = compute_metrics(y_val, torch.sigmoid(val_logits).numpy())['macro F1']
        val_loss = nn.functional.binary_cross_entropy_with_logits(
            val_logits, torch.tensor(y_val), pos_weight=pos_weight.float()).item()
        train_f1 = compute_metrics(y_train_orig, predict_proba(model, train_eval_loader))['macro F1']
        history.append({'epoch': epoch, 'train_loss': epoch_loss / len(train_dataset), 'val_macro_f1': val_f1,
                        'val_loss': val_loss, 'train_macro_f1': train_f1})
        if val_f1 > best_f1:
            best_f1, best_epoch = val_f1, epoch
            best_state = {key: value.cpu().clone() for key, value in model.state_dict().items()}

    model.load_state_dict(best_state)
    val_probs = predict_proba(model, val_loader)
    return {
        'seed': seed,
        'best_epoch': best_epoch,
        'history': history,
        'val_probs': val_probs,
        'test_probs': predict_proba(model, test_loader),
        'thresholds': tune_thresholds(y_val, val_probs),
    }

In [6]:
TRAIN_MODELS = False
RUNS_FILE = ROOT / 'Experiments' / 'results' / 'task5_overfitting_curves.pkl'
MODEL_LAYERS = {'Baseline': 1, 'Stacked': 2}

if TRAIN_MODELS:
    runs = {name: [] for name in MODEL_LAYERS}
    for name, num_layers in MODEL_LAYERS.items():
        for seed in SEEDS:
            start = time.time()
            run = train_model(num_layers, seed)
            runs[name].append(run)
            best = run['history'][run['best_epoch'] - 1]
            print(f"{name} | seed {seed} | best epoch {run['best_epoch']:2d} | val macro F1 {best['val_macro_f1']:.4f} | "
                  f"train macro F1 {best['train_macro_f1']:.4f} | {time.time() - start:.0f} s")
    pd.to_pickle(runs, RUNS_FILE)
else:
    runs = pd.read_pickle(RUNS_FILE)

Baseline | seed 42 | best epoch 16 | val macro F1 0.5453 | train macro F1 0.9161 | 278 s


Baseline | seed 1337 | best epoch 17 | val macro F1 0.5399 | train macro F1 0.8957 | 299 s


Baseline | seed 2026 | best epoch 16 | val macro F1 0.5518 | train macro F1 0.8764 | 263 s


Stacked | seed 42 | best epoch 12 | val macro F1 0.5591 | train macro F1 0.8337 | 389 s


Stacked | seed 1337 | best epoch 18 | val macro F1 0.5542 | train macro F1 0.9240 | 347 s


Stacked | seed 2026 | best epoch 14 | val macro F1 0.5275 | train macro F1 0.8127 | 399 s


In [7]:
reference = pd.read_pickle(ROOT / 'Experiments' / 'results' / 'task5_runs.pkl')
print('Reproduction of Task 5 (best epoch, val macro F1 at the best epoch):')
for name in MODEL_LAYERS:
    for new_run, old_run in zip(runs[name], reference[name]):
        print(f"  {name} seed {new_run['seed']}: new {new_run['best_epoch']}, "
              f"{new_run['history'][new_run['best_epoch'] - 1]['val_macro_f1']:.4f} | "
              f"Task 5 {old_run['best_epoch']}, {old_run['history'][old_run['best_epoch'] - 1]['val_macro_f1']:.4f}")
print()
for name in MODEL_LAYERS:
    curves = pd.DataFrame([h | {'seed': run['seed']} for run in runs[name] for h in run['history']])
    mean = curves.drop(columns='seed').groupby('epoch').mean()
    print(name, '(mean over seeds)')
    print(mean[['train_loss', 'val_loss', 'train_macro_f1', 'val_macro_f1']].round(4).to_string())
    best = pd.DataFrame([run['history'][run['best_epoch'] - 1] for run in runs[name]])
    last = pd.DataFrame([run['history'][-1] for run in runs[name]])
    min_val_loss = [int(np.argmin([h['val_loss'] for h in run['history']])) + 1 for run in runs[name]]
    print(f"  at best epoch: train F1 {best['train_macro_f1'].mean():.3f}, val F1 {best['val_macro_f1'].mean():.3f}")
    print(f"  at epoch 20:   train F1 {last['train_macro_f1'].mean():.3f}, val F1 {last['val_macro_f1'].mean():.3f}")
    print(f"  epoch of minimum validation loss per seed: {min_val_loss}")
    print()

Reproduction of Task 5 (best epoch, val macro F1 at the best epoch):
  Baseline seed 42: new 16, 0.5453 | Task 5 16, 0.5453
  Baseline seed 1337: new 17, 0.5399 | Task 5 17, 0.5399
  Baseline seed 2026: new 16, 0.5518 | Task 5 16, 0.5518
  Stacked seed 42: new 12, 0.5591 | Task 5 12, 0.5591
  Stacked seed 1337: new 18, 0.5542 | Task 5 18, 0.5542
  Stacked seed 2026: new 14, 0.5275 | Task 5 14, 0.5275

Baseline (mean over seeds)
       train_loss  val_loss  train_macro_f1  val_macro_f1
epoch                                                    
1          0.8067    0.5627          0.3076        0.2878
2          0.3504    0.4845          0.3391        0.3129
3          0.2499    0.4727          0.3863        0.3356
4          0.1813    0.5514          0.4628        0.3831
5          0.1344    0.6240          0.5374        0.4251
6          0.1103    0.7269          0.5674        0.4302
7          0.0843    0.8797          0.6413        0.4701
8          0.0686    0.9381          0.6718   